# Supply Chain Risk Modelling — Phase 1: Data Preparation

This notebook is the entry point for the whole pipeline. The goal is to take the raw WorldRiskIndex (WRI) dataset — 193 countries, 26 years, 248 columns — understand its structure properly, clean up the naming conventions, and produce three tidy output tables that'll be used for Feature Engineering and ML model development.

The WRI decomposes national risk into Exposure (natural hazard intensity) and Vulnerability (the capacity to absorb and recover from disruption). What I'm building on top of it is a supply chain routing model which answers how reliable a route is as a logistics corridor? This drives every feature engineering decision downstream.

In [3]:
# Core data manipulation
import pandas as pd
import numpy as np

# Visualization 
import matplotlib.pyplot as plt
import seaborn as sns

# Utility
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

In [4]:
wri_trend = pd.read_csv("worldriskindex-trend.csv")

# Metadata
wri_meta = pd.read_excel("worldriskindex-meta.xlsx")

print("WRI Trend shape:", wri_trend.shape)
print("WRI Meta shape:", wri_meta.shape)

WRI Trend shape: (5018, 248)
WRI Meta shape: (151, 9)


## Understanding the WRI Structure

The WRI is a three-level tree — raw indicators aggregate upward through domain scores, components, and composites to a single country score. Each row has the raw values (`_Base` columns), their 0–100 normalized counterparts (`_Norm` columns), and all intermediate aggregates.

For feature engineering, I'll work exclusively with the `_Norm` columns and domain-level scores — not the top-level composites like `W`, `E`, or `V`, which would introduce multicollinearity into any model that also uses their constituent parts.

In [6]:
print("=" * 60)
print("METADATA OVERVIEW")
print("=" * 60)
wri_meta.head(10)

METADATA OVERVIEW


,Code,Variable,Type,Unit,Latest Year,Data Provider,Data Codes,Last Access,URL
0,Country,Country Name,-,-,-,United Nations Statistics Division,-,2024-04-30,https://unstats.un.org/unsd/methodology/m49/
1,ISO3,Three Digit Country Code,-,-,-,United Nations Statistics Division,-,2024-04-30,https://unstats.un.org/unsd/methodology/m49/
2,W,WorldRiskIndex,Index,NaN,NaN,NaN,Geometric Mean of E And V,NaT,NaN
3,E,Exposition,Sphere,NaN,NaN,Own Calculations,"Geometric Mean Of EI_01, EI_02, EI_03, EI_04, ...",NaT,NaN
4,EI_01,Earthquakes,Domain,NaN,NaN,Own Calculations,"Geometric Mean Of EI_01a, EI_01b, EI_01c, EI_0...",NaT,NaN
5,EI_01a,Annually Averaged Population Exposed To Strong...,Indicator,Number,2020,Global Facility For Disaster Reduction And Rec...,Weighted Mean Of Sums Of Affected Raster Cell ...,2024-04-30,https://www.geonode-gfdrrlab.org/; https://www...
6,EI_01b,Annually Averaged Population Exposed To Strong...,Indicator,Percent,2020,Global Facility For Disaster Reduction And Rec...,Weighted Mean Of Sums Of Affected Raster Cell ...,2024-04-30,https://www.geonode-gfdrrlab.org/; https://www...
7,EI_01c,Annually Averaged Population Exposed To Severe...,Indicator,Number,2020,Global Facility For Disaster Reduction And Rec...,Weighted Mean Of Sums Of Affected Raster Cell ...,2024-04-30,https://www.geonode-gfdrrlab.org/; https://www...
8,EI_01d,Annually Averaged Population Exposed To Severe...,Indicator,Percent,2020,Global Facility For Disaster Reduction And Rec...,Weighted Mean Of Sums Of Affected Raster Cell ...,2024-04-30,https://www.geonode-gfdrrlab.org/; https://www...
9,EI_01e,Annually Averaged Population Exposed To Extrem...,Indicator,Number,2020,Global Facility For Disaster Reduction And Rec...,Weighted Mean Of Sums Of Affected Raster Cell ...,2024-04-30,https://www.geonode-gfdrrlab.org/; https://www...


In [7]:
print("WRI VARIABLE HIERARCHY (from metadata)")
print("\nVariable types in the dataset:")
print(wri_meta['Type'].value_counts().to_string())

meta_lookup = wri_meta.set_index('Code')[['Variable', 'Type', 'Unit']].to_dict('index')

WRI VARIABLE HIERARCHY (from metadata)

Variable types in the dataset:
Type
Indicator    100
Domain        32
Component     11
Dimension      3
-              2
Sphere         2
Index          1


In [8]:
print("=" * 60)
print("TREND DATA OVERVIEW")
print("=" * 60)
wri_trend.head(10)

TREND DATA OVERVIEW


,WRI.Country,ISO3.Code,Year,W,E,V,S,C,A,S_01,S_02,S_03,S_04,S_05,C_01,C_02,C_03,A_01,A_02,A_03,EI_01,EI_02,EI_03,EI_04,EI_05,EI_06,EI_07,SI_01,SI_02,SI_03,SI_04,SI_05,SI_06,SI_07,SI_08,SI_09,SI_10,SI_11,SI_12,SI_13,SI_14,CI_01,CI_02,CI_03,CI_04,CI_05,CI_06,CI_07,AI_01,AI_02,AI_03,AI_04,EI_01a_Norm,EI_01a_Base,EI_01b_Norm,EI_01b_Base,EI_01c_Norm,EI_01c_Base,EI_01d_Norm,EI_01d_Base,EI_01e_Norm,EI_01e_Base,EI_01f_Norm,EI_01f_Base,EI_02a_Norm,EI_02a_Base,EI_02b_Norm,EI_02b_Base,EI_02c_Norm,EI_02c_Base,EI_02d_Norm,EI_02d_Base,EI_02e_Norm,EI_02e_Base,EI_02f_Norm,EI_02f_Base,EI_03a_Norm,EI_03a_Base,EI_03b_Norm,EI_03b_Base,EI_03c_Norm,EI_03c_Base,EI_03d_Norm,EI_03d_Base,EI_03e_Norm,EI_03e_Base,EI_03f_Norm,EI_03f_Base,EI_04a_Norm,EI_04a_Base,EI_04b_Norm,EI_04b_Base,EI_04c_Norm,EI_04c_Base,EI_04d_Norm,EI_04d_Base,EI_04e_Norm,EI_04e_Base,EI_04f_Norm,EI_04f_Base,EI_05a_Norm,EI_05a_Base,EI_05b_Norm,EI_05b_Base,EI_05c_Norm,EI_05c_Base,EI_05d_Norm,EI_05d_Base,EI_05e_Norm,EI_05e_Base,EI_05f_Norm,EI_05f_Base,EI_06a_Norm,EI_06a_Base,EI_06b_Norm,EI_06b_Base,EI_06c_Norm,EI_06c_Base,EI_06d_Norm,EI_06d_Base,EI_06e_Norm,EI_06e_Base,EI_06f_Norm,EI_06f_Base,EI_07a_Norm,EI_07a_Base,EI_07b_Norm,EI_07b_Base,SI_01a_Norm,SI_01a_Base,SI_01b_Norm,SI_01b_Base,SI_02a_Norm,SI_02a_Base,SI_02b_Norm,SI_02b_Base,SI_03a_Norm,SI_03a_Base,SI_03b_Norm,SI_03b_Base,SI_04a_Norm,SI_04a_Base,SI_04b_Norm,SI_04b_Base,SI_05a_Norm,SI_05a_Base,SI_05b_Norm,SI_05b_Base,SI_06a_Norm,SI_06a_Base,SI_06b_Norm,SI_06b_Base,SI_07a_Norm,SI_07a_Base,SI_07b_Norm,SI_07b_Base,SI_08a_Norm,SI_08a_Base,SI_08b_Norm,SI_08b_Base,SI_09a_Norm,SI_09a_Base,SI_09b_Norm,SI_09b_Base,SI_10_Norm,SI_10_Base,SI_10a_Norm,SI_10a_Base,SI_10b_Norm,SI_10b_Base,SI_11_Norm,SI_11_Base,SI_12a_Norm,SI_12a_Base,SI_12b_Norm,SI_12b_Base,SI_13a_Norm,SI_13a_Base,SI_13b_Norm,SI_13b_Base,SI_14a_Norm,SI_14a_Base,SI_14b_Norm,SI_14b_Base,SI_15a_Norm,SI_15a_Base,SI_15b_Norm,SI_15b_Base,SI_15c_Norm,SI_15c_Base,SI_15d_Norm,SI_15d_Base,CI_01a_Norm,CI_01a_Base,CI_01b_Norm,CI_01b_Base,CI_02a_Norm,CI_02a_Base,CI_02b_Norm,CI_02b_Base,CI_03a_Norm,CI_03a_Base,CI_03b_Norm,CI_03b_Base,CI_04a_Norm,CI_04a_Base,CI_04b_Norm,CI_04b_Base,CI_05a_Norm,CI_05a_Base,CI_05b_Norm,CI_05b_Base,CI_06a_Norm,CI_06a_Base,CI_06b_Norm,CI_06b_Base,CI_07a_Norm,CI_07a_Base,CI_07b_Norm,CI_07b_Base,AI_01a_Norm,AI_01a_Base,AI_01b_Norm,AI_01b_Base,AI_01c_Norm,AI_01c_Base,AI_02a_Norm,AI_02a_Base,AI_02b_Norm,AI_02b_Base,AI_02c_Norm,AI_02c_Base,AI_03a_Norm,AI_03a_Base,AI_03b_Norm,AI_03b_Base,AI_03c_Norm,AI_03c_Base,AI_04a_Norm,AI_04a_Base,AI_04b_Norm,AI_04b_Base,AI_04c_Norm,AI_04c_Base,AI_05a_Norm,AI_05a_Base,AI_05b_Norm,AI_05b_Base
0,Afghanistan,AFG,2000,4.18,0.25,69.83,61.97,73.22,75.05,64.24,75.39,68.60,59.19,46.47,59.34,89.75,73.72,66.77,87.95,71.97,56.79,0.01,0.01,44.91,0.01,2.49,0.01,69.30,81.03,76.79,39.50,70.18,67.71,77.73,87.45,38.52,93.27,89.84,64.46,53.77,59.83,42.57,82.71,89.19,90.31,69.52,73.69,78.22,64.41,69.22,79.67,97.10,65.22,117738.67,66.24,0.58,59.54,39680.72,53.79,0.20,52.07,1423.12,46.58,0.01,0.01,0.00,0.01,0.00,0.01,0.00,0.01,0.00,0.01,0.00,0.01,0.00,0.01,0.00,0.01,0.00,0.01,0.00,0.01,0.00,0.01,0.00,0.01,0.00,48.61,204241.40,40.13,1.01,48.20,172429.40,40.47,0.85,50.28,131433.28,42.91,0.65,0.01,0.00,0.01,0.00,0.01,0.00,0.01,0.00,0.01,0.00,0.01,0.00,55.86,1203634.53,40.55,5.95,34.38,20911.21,30.44,0.10,0.01,0.00,0.01,0.00,0.01,0.00,0.01,0.00,67.55,55.00,71.09,9.31,87.45,1.26,75.09,5.86,77.87,788.07,75.73,-259.25,43.10,31.42,36.20,-15.52,81.49,27.44,60.44,20.97,77.41,4.40,59.22,1.11,60.42,0.73,100.00,0.00,87.46,51.71,87.45,83.55,39.97,52.45,37.12,233.00,93.27,109.59,93.27,104.85,27.18,4.74,89.84,0.49,70.83,298040,58.66,1.48,60.49,95853.69,47.80,0.26,62.04,274840.27,57.70,0.77,29.14,5.30,44.82,29847.56,58.96,38362.52,60.57,2515.26,45.01,55273.60,40.27,0.28,88.05,5138.60,77.70,0.03,79.55,-1.27,100.00,-2.17,93.27,-2.44,87.45,-1.78,60.03,28.21,80.51,26.32,87.45,40.82,62.09,4.29,87.45,1346.14,69.97,131.70,70.84,0.52,46.26,51.67,81.55,40.52,66.60,0.10,72.05,0

In [9]:
# The WRI score is built bottom-up from ~193 leaf indicators.
# Mapping this tree explicitly prevents the common mistake of using
# parent and child scores together (double-counting the same signal).
#
# W (WorldRiskIndex) = Geometric Mean of:
#   ├── E (Exposure)          → 7 domains (EI_01..EI_07): earthquakes, tsunamis,
#   │                            coastal floods, riverine floods, cyclones, droughts,
#   │                            sea level rise
#   └── V (Vulnerability)     = Geometric Mean of:
#       ├── S (Susceptibility) → 5 components (S_01..S_05)
#       │     ├── S_01: Socio-Economic Development    → SI_01..SI_04
#       │     ├── S_02: Socio-Economic Deprivation    → SI_05..SI_08
#       │     ├── S_03: Societal Disparities          → SI_09..SI_11
#       │     ├── S_04: Vulnerable Pops (Violence)    → SI_12..SI_14
#       │     └── S_05: Vulnerable Pops (Disease)     → SI_15a..SI_15d
#       ├── C (Lack of Coping) → 3 components (C_01..C_03)
#       │     ├── C_01: Recent Societal Shocks        → CI_01..CI_02
#       │     ├── C_02: State and Government          → CI_03..CI_04
#       │     └── C_03: Health Care Capacities        → CI_05..CI_07
#       └── A (Lack of Adaptive Capacity) → 3 components (A_01..A_03)
#             ├── A_01: Research and Education         → AI_01..AI_02
#             ├── A_02: Long-Term Health/Deprivation   → AI_03..AI_04
#             └── A_03: Investment Capacities          → AI_05a..AI_05b

print("\nThe hierarchy above is the analytical backbone of our supply chain model.")
print("Every Indicator (leaf node) appears in TWO forms in the dataset:")
print("  - _Norm: Normalized score (0-100 scale, comparable across countries)")
print("  - _Base: Raw value (original units — population counts, percentages, USD, etc.)")


The hierarchy above is the analytical backbone of our supply chain model.
Every Indicator (leaf node) appears in TWO forms in the dataset:
  - _Norm: Normalized score (0-100 scale, comparable across countries)
  - _Base: Raw value (original units — population counts, percentages, USD, etc.)


In [10]:
# --- Categorize all 248 columns ---

# Row identifiers
ID_COLS = ['WRI.Country', 'ISO3.Code', 'Year']

# Top-level composites — derived, not raw. Don't use alongside their sub-components.
TOP_SCORES = ['W', 'E', 'V', 'S', 'C', 'A']

# Mid-level component scores (sit between composites and domain indicators)
COMPONENT_COLS = [c for c in wri_trend.columns 
                  if c not in ID_COLS + TOP_SCORES 
                  and '_' not in c]


# Domain scores — the granularity level used for feature engineering
DOMAIN_COLS = [c for c in wri_trend.columns 
               if '_' in c 
               and '_Norm' not in c 
               and '_Base' not in c]

# Normalized indicator scores (0-100, what we'll primarily model with)
NORM_COLS = [c for c in wri_trend.columns if '_Norm' in c]

# Raw base values (original units — useful for interpretability)
BASE_COLS = [c for c in wri_trend.columns if '_Base' in c]

print(f"\nColumn breakdown across {len(wri_trend.columns)} total columns:")
print(f"  Identity columns:         {len(ID_COLS)}")
print(f"  Top-level scores (W,E,V,S,C,A): {len(TOP_SCORES)}")
print(f"  Domain/Component scores:  {len(DOMAIN_COLS)}")
print(f"  Normalized indicators:    {len(NORM_COLS)}")
print(f"  Raw base values:          {len(BASE_COLS)}")
print(f"  Total:                    {len(ID_COLS)+len(TOP_SCORES)+len(DOMAIN_COLS)+len(NORM_COLS)+len(BASE_COLS)}")


Column breakdown across 248 total columns:
  Identity columns:         3
  Top-level scores (W,E,V,S,C,A): 6
  Domain/Component scores:  43
  Normalized indicators:    98
  Raw base values:          98
  Total:                    248


## Exploration

A few things I needed to verify before building any features: Is this a balanced dataset (every country in every year)? Are there true missing values or just true zeros representing "no exposure"? How much do WRI scores move year-to-year — because that determines whether rolling volatility features will carry any signal. And do Exposure and Vulnerability move together, or do they provide genuinely independent information?

In [12]:
# --- 3A: Temporal and geographic scope ---
print("=" * 70)
print("DATASET SCOPE")
print("=" * 70)
print(f"Time range: {wri_trend['Year'].min()} to {wri_trend['Year'].max()}")
print(f"Years covered: {wri_trend['Year'].nunique()} consecutive years")
print(f"Countries: {wri_trend['WRI.Country'].nunique()}")
print(f"Total observations: {len(wri_trend)} (should be countries × years)")

# Verify it's a balanced panel (every country has every year)
expected = wri_trend['WRI.Country'].nunique() * wri_trend['Year'].nunique()
actual = len(wri_trend)
print(f"\nExpected rows (193 × 26): {expected}")
print(f"Actual rows:              {actual}")
print(f"Balanced panel: {'YES' if expected == actual else 'NO — investigate gaps'}")

DATASET SCOPE
Time range: 2000 to 2025
Years covered: 26 consecutive years
Countries: 193
Total observations: 5018 (should be countries × years)

Expected rows (193 × 26): 5018
Actual rows:              5018
Balanced panel: YES


In [13]:
# --- 3B: Missing values and data completeness ---
print("\n" + "=" * 70)
print("DATA COMPLETENESS CHECK")
print("=" * 70)

total_cells = wri_trend.shape[0] * wri_trend.shape[1]
total_missing = wri_trend.isnull().sum().sum()
print(f"Total cells: {total_cells:,}")
print(f"Missing cells: {total_missing}")

if total_missing == 0:
    print("\nZero missing values across all 248 columns.")
    print("Unusually clean for a public dataset — WRI likely imputes before publishing.")
    print("No imputation needed on our end, but worth checking for SENTINEL values (like 0.01)")
    print("that might be stand-ins for 'no data'.")


DATA COMPLETENESS CHECK
Total cells: 1,244,464
Missing cells: 0

Zero missing values across all 248 columns.
Unusually clean for a public dataset — WRI likely imputes before publishing.
No imputation needed on our end, but worth checking for SENTINEL values (like 0.01)
that might be stand-ins for 'no data'.


In [14]:
# --- 3C: Sentinel value check ---
# The WRI uses 0.01 (not NaN) for near-zero exposure — e.g., landlocked countries
# with negligible tsunami risk. Worth quantifying how many cells hit this floor
# before treating it as genuine missing data in any downstream model.

print("\n" + "=" * 70)
print("SENTINEL VALUE ANALYSIS")
print("=" * 70)

# Check how many cells equal exactly 0.01 in the score columns
score_cols = TOP_SCORES + DOMAIN_COLS + NORM_COLS + BASE_COLS
sentinel_count = (wri_trend[score_cols] == 0.01).sum()
high_sentinel = sentinel_count[sentinel_count > 100].sort_values(ascending=False)

print(f"Columns with >100 cells equal to exactly 0.01:")
for col, count in high_sentinel.items():
    pct = count / len(wri_trend) * 100
    desc = meta_lookup.get(col.replace('_Norm','').replace('_Base',''), {})
    var_name = desc.get('Variable', '')[:60] if desc else ''
    print(f"  {col:20s}: {count:5d} ({pct:5.1f}%)  {var_name}")


SENTINEL VALUE ANALYSIS
Columns with >100 cells equal to exactly 0.01:
  EI_05e_Norm         :  4758 ( 94.8%)  Annually Averaged Population Exposed To Extreme Intensity (S
  EI_05f_Norm         :  4758 ( 94.8%)  Annually Averaged Population Exposed To Extreme Intensity (S
  SI_14b_Norm         :  4103 ( 81.8%)  Internally Displaced Persons Due To Violence And Conflict
  CI_02b_Norm         :  3823 ( 76.2%)  Average Population Killed In Conflicts In The Last 5 Years
  SI_14               :  3815 ( 76.0%)  Internally Displaced Persons Due To Violence And Conflict
  SI_14a_Norm         :  3815 ( 76.0%)  Internally Displaced Persons Due To Violence And Conflict
  CI_02a_Norm         :  3781 ( 75.3%)  Average Population Killed In Conflicts In The Last 5 Years
  CI_02               :  3781 ( 75.3%)  Recent Societal Shocks Due To Violence And Conflicts
  EI_06f_Norm         :  3770 ( 75.1%)  Annually Averaged Population Exposed To Extreme Intensity (6
  EI_05d_Norm         :  3770 ( 75.1%)  

In [15]:
# --- 3D: Distribution of the main WRI score ---
print("\n" + "=" * 70)
print("WRI SCORE DISTRIBUTION (W = WorldRiskIndex)")
print("=" * 70)

wri_scores = wri_trend['W']
print(f"Mean:   {wri_scores.mean():.2f}")
print(f"Median: {wri_scores.median():.2f}")
print(f"Std:    {wri_scores.std():.2f}")
print(f"Min:    {wri_scores.min():.2f} ({wri_trend.loc[wri_scores.idxmin(), 'WRI.Country']})")
print(f"Max:    {wri_scores.max():.2f} ({wri_trend.loc[wri_scores.idxmax(), 'WRI.Country']})")
print(f"Skew:   {wri_scores.skew():.2f}")

# Distribution shape tells us which models will work well
if wri_scores.skew() > 1:
    print("\n→ Right-skewed: most countries have low-moderate risk,")
    print("  a long tail of high-risk countries. Tree-based models")
    print("  handle this well; linear models may need log-transform.")


WRI SCORE DISTRIBUTION (W = WorldRiskIndex)
Mean:   8.20
Median: 4.19
Std:    9.23
Min:    0.16 (Monaco)
Max:    47.42 (Philippines)
Skew:   1.93

→ Right-skewed: most countries have low-moderate risk,
  a long tail of high-risk countries. Tree-based models
  handle this well; linear models may need log-transform.


In [16]:
# --- 3E: Top and bottom risk countries (latest year) ---
latest_year = wri_trend['Year'].max()
latest = wri_trend[wri_trend['Year'] == latest_year].copy()

print(f"\n{'=' * 70}")
print(f"HIGHEST AND LOWEST RISK COUNTRIES ({latest_year})")
print(f"{'=' * 70}")

display_cols = ['WRI.Country', 'ISO3.Code', 'W', 'E', 'V', 'S', 'C', 'A']

print("\nTop 15 highest-risk countries:")
top15 = latest.nlargest(15, 'W')[display_cols].reset_index(drop=True)
top15.index = top15.index + 1
print(top15.to_string())

print("\nTop 10 lowest-risk countries:")
bot10 = latest.nsmallest(10, 'W')[display_cols].reset_index(drop=True)
bot10.index = bot10.index + 1
print(bot10.to_string())


HIGHEST AND LOWEST RISK COUNTRIES (2025)

Top 15 highest-risk countries:
           WRI.Country ISO3.Code     W     E     V     S     C     A
1          Philippines       PHL 46.56 39.99 54.20 50.10 58.54 54.30
2                India       IND 40.73 35.99 46.10 34.56 54.08 52.43
3            Indonesia       IDN 39.80 39.89 39.71 24.80 51.27 49.24
4             Colombia       COL 39.26 31.54 48.86 47.85 50.87 47.91
5               Mexico       MEX 38.96 50.08 30.31 44.39 12.53 50.07
6              Myanmar       MMR 36.91 22.43 60.74 55.42 64.47 62.72
7           Mozambique       MOZ 34.39 18.10 65.33 65.91 63.33 66.79
8   Russian Federation       RUS 31.22 28.35 34.38 26.49 39.99 38.36
9                China       CHN 30.62 64.59 14.52  8.96 11.44 29.85
10            Pakistan       PAK 26.82 13.11 54.85 40.52 63.30 64.34
11          Bangladesh       BGD 26.71 16.57 43.07 27.09 59.29 49.73
12    Papua New Guinea       PNG 26.51 18.84 37.29 57.52 13.36 67.46
13            Viet Nam       

In [17]:
# --- 3F: Year-over-year movement ---
# Checking how much scores shift annually — if WRI scores barely move,
# rolling volatility features won't carry much signal worth modelling.

print(f"\n{'=' * 70}")
print("YEAR-OVER-YEAR SCORE STABILITY")
print(f"{'=' * 70}")

wri_sorted = wri_trend.sort_values(['WRI.Country', 'Year'])
wri_sorted['W_yoy_change'] = wri_sorted.groupby('WRI.Country')['W'].diff()
wri_sorted['W_yoy_pct_change'] = wri_sorted.groupby('WRI.Country')['W'].pct_change() * 100

yoy = wri_sorted['W_yoy_change'].dropna()
print(f"Mean absolute YoY change:   {yoy.abs().mean():.3f}")
print(f"Median absolute YoY change: {yoy.abs().median():.3f}")
print(f"Max YoY increase:           {yoy.max():.3f}")
print(f"Max YoY decrease:           {yoy.min():.3f}")

# Find countries with the most volatile WRI scores
country_vol = wri_sorted.groupby('WRI.Country')['W_yoy_change'].std()
print(f"\nMost volatile countries (highest std of YoY change):")
for country, vol in country_vol.nlargest(10).items():
    print(f"  {country:30s}: std = {vol:.3f}")


YEAR-OVER-YEAR SCORE STABILITY
Mean absolute YoY change:   0.377
Median absolute YoY change: 0.110
Max YoY increase:           8.960
Max YoY decrease:           -11.720

Most volatile countries (highest std of YoY change):
  Mexico                        : std = 3.632
  Japan                         : std = 2.925
  Indonesia                     : std = 2.798
  Bangladesh                    : std = 2.346
  Papua New Guinea              : std = 2.332
  China                         : std = 2.289
  Peru                          : std = 2.246
  Ecuador                       : std = 2.169
  Iran (Islamic Republic of)    : std = 2.055
  Yemen                         : std = 1.978


In [18]:
# --- 3G: Correlation between top-level dimensions ---
# If Exposure and Vulnerability are tightly correlated, separating them
# adds little value. Weak correlation means they're telling different stories —
# and that's the whole point of a multi-dimensional model.

print(f"\n{'=' * 70}")
print("CORRELATION BETWEEN RISK DIMENSIONS")
print(f"{'=' * 70}")

corr_cols = ['W', 'E', 'V', 'S', 'C', 'A']
corr_matrix = wri_trend[corr_cols].corr().round(3)
print(corr_matrix.to_string())

print("\nKey insight: If E and V are weakly correlated, a country can be")
print("highly exposed to natural hazards but resilient (low vulnerability),")
print("or low-exposure but fragile. This decomposition is what makes WRI")
print("useful for supply chain analysis — exposure alone doesn't tell the story.")


CORRELATION BETWEEN RISK DIMENSIONS
     W     E    V    S    C     A
W 1.00  0.88 0.40 0.33 0.42  0.21
E 0.88  1.00 0.14 0.11 0.18 -0.00
V 0.40  0.14 1.00 0.90 0.90  0.75
S 0.33  0.11 0.90 1.00 0.69  0.70
C 0.42  0.18 0.90 0.69 1.00  0.50
A 0.21 -0.00 0.75 0.70 0.50  1.00

Key insight: If E and V are weakly correlated, a country can be
highly exposed to natural hazards but resilient (low vulnerability),
or low-exposure but fragile. This decomposition is what makes WRI
useful for supply chain analysis — exposure alone doesn't tell the story.


## Cleaning

The WRI data is unusually clean — zero missing values across 248 columns, which suggests the WRI team pre-imputes or drops incomplete observations before publishing. The cleaning work here is mostly structural: standardising column names, enforcing data types, and organising columns into named lists so that feature enginering code can reference the right tier of indicators without manual lookup every time.

In [20]:
# Make a working copy so we preserve the raw data
wri_clean = wri_trend.copy()

# --- 4A: Column name standardisation ---
# Original names use dot-notation (WRI.Country) which breaks .query() and
# attribute-style access. Snake_case throughout from here on.

wri_clean = wri_clean.rename(columns={
    'WRI.Country': 'country',
    'ISO3.Code': 'iso3',
    'Year': 'year'
})

# Lowercase all remaining column names for consistency
wri_clean.columns = wri_clean.columns.str.lower()

print("=" * 70)
print("COLUMN STANDARDIZATION")
print("=" * 70)
print(f"Renamed: WRI.Country → country, ISO3.Code → iso3, Year → year")
print(f"All columns lowercased.")
print(f"Sample: {wri_clean.columns[:12].tolist()}")

COLUMN STANDARDIZATION
Renamed: WRI.Country → country, ISO3.Code → iso3, Year → year
All columns lowercased.
Sample: ['country', 'iso3', 'year', 'w', 'e', 'v', 's', 'c', 'a', 's_01', 's_02', 's_03']


In [21]:
# --- 4B: Rebuild column categories with cleaned names ---

ID_COLS = ['country', 'iso3', 'year']

# Top-level composite indices (W = f(E,V); V = f(S,C,A))
COMPOSITE_SCORES = ['w', 'e', 'v', 's', 'c', 'a']

# Component-level scores (S_01..S_05, C_01..C_03, A_01..A_03)
COMPONENT_COLS = ['s_01', 's_02', 's_03', 's_04', 's_05',
                  'c_01', 'c_02', 'c_03',
                  'a_01', 'a_02', 'a_03']

# Domain-level scores (EI_01..EI_07, SI_01..SI_14, CI_01..CI_07, AI_01..AI_04)
DOMAIN_COLS = [c for c in wri_clean.columns 
               if ('ei_' in c or 'si_' in c or 'ci_' in c or 'ai_' in c)
               and '_norm' not in c and '_base' not in c
               and c not in COMPONENT_COLS]

# Normalized indicators (0-100 scale) — PRIMARY modeling features
NORM_COLS = [c for c in wri_clean.columns if '_norm' in c]

# Raw base values (original units) — for interpretability only
BASE_COLS = [c for c in wri_clean.columns if '_base' in c]

ANALYSIS_COLS = COMPOSITE_SCORES + COMPONENT_COLS + DOMAIN_COLS

print(f"\nColumn categories (cleaned names):")
print(f"  ID columns:          {len(ID_COLS)}")
print(f"  Composite scores:    {len(COMPOSITE_SCORES)}")
print(f"  Component scores:    {len(COMPONENT_COLS)}")
print(f"  Domain scores:       {len(DOMAIN_COLS)}")
print(f"  Norm indicators:     {len(NORM_COLS)}")
print(f"  Base indicators:     {len(BASE_COLS)}")


Column categories (cleaned names):
  ID columns:          3
  Composite scores:    6
  Component scores:    11
  Domain scores:       32
  Norm indicators:     98
  Base indicators:     98


In [22]:
# --- 4C: Data type enforcement ---
wri_clean['year'] = wri_clean['year'].astype(int)
wri_clean['country'] = wri_clean['country'].astype(str).str.strip()
wri_clean['iso3'] = wri_clean['iso3'].astype(str).str.strip()

all_score_cols = COMPOSITE_SCORES + COMPONENT_COLS + DOMAIN_COLS + NORM_COLS + BASE_COLS
for col in all_score_cols:
    wri_clean[col] = pd.to_numeric(wri_clean[col], errors='coerce')

print(f"\nData types enforced:")
print(f"  year: {wri_clean['year'].dtype}")
print(f"  country: {wri_clean['country'].dtype}")
print(f"  Score columns: all float64")


Data types enforced:
  year: int64
  country: object
  Score columns: all float64


In [23]:
# --- 4D: Verification — confirm no damage from cleaning ---
print(f"\n{'=' * 70}")
print("POST-CLEANING VERIFICATION")
print(f"{'=' * 70}")

print(f"Shape: {wri_clean.shape} (should be 5018 × 248)")
print(f"Missing values introduced: {wri_clean.isnull().sum().sum()}")
print(f"Duplicates introduced: {wri_clean.duplicated(subset=['country', 'year']).sum()}")
print(f"Year range preserved: {wri_clean['year'].min()} – {wri_clean['year'].max()}")
print(f"Country count preserved: {wri_clean['country'].nunique()}")

# Quick spot-check: Afghanistan 2000 should have W ≈ 4.18
afg_2000 = wri_clean[(wri_clean['country'] == 'Afghanistan') & (wri_clean['year'] == 2000)]
print(f"\nSpot check — Afghanistan 2000 WRI: {afg_2000['w'].values[0]:.2f} (expected: 4.18)")


POST-CLEANING VERIFICATION
Shape: (5018, 248) (should be 5018 × 248)
Missing values introduced: 0
Duplicates introduced: 0
Year range preserved: 2000 – 2025
Country count preserved: 193

Spot check — Afghanistan 2000 WRI: 4.18 (expected: 4.18)


## Output Tables

Three tables come out of this notebook, each serving a specific purpose downstream.

The **Core Risk Panel** is the full time series trimmed to the columns actually used in feature engineering — this is what Feature engineering loads. 

The **Latest-Year Snapshot** is a cross-sectional slice at the most recent year, with global ranks added for each composite score — useful for the dashboard and visualisation layer. 

The **Risk Decomposition Summary** aggregates each country's time-series statistics (mean risk levels, volatility, trend direction) into a single row per country — the visualisation notebook pulls from this directly for several charts.

In [25]:
# Ensure sorted by country then year (critical for time-series ops)
wri_clean = wri_clean.sort_values(['iso3', 'year']).reset_index(drop=True)

core_cols = ID_COLS + COMPOSITE_SCORES + COMPONENT_COLS + DOMAIN_COLS
wri_core_panel = wri_clean[core_cols].copy()

print("=" * 70)
print("TABLE 1: CORE RISK PANEL (for feature engineering)")
print("=" * 70)
print(f"Shape: {wri_core_panel.shape}")
print(f"Columns: {wri_core_panel.columns.tolist()[:15]}... ({len(wri_core_panel.columns)} total)")
print(f"\nSample — Afghanistan first 3 years:")
print(wri_core_panel[wri_core_panel['country'] == 'Afghanistan'].head(3).to_string())

TABLE 1: CORE RISK PANEL (for feature engineering)
Shape: (5018, 52)
Columns: ['country', 'iso3', 'year', 'w', 'e', 'v', 's', 'c', 'a', 's_01', 's_02', 's_03', 's_04', 's_05', 'c_01']... (52 total)

Sample — Afghanistan first 3 years:
       country iso3  year    w    e     v     s     c     a  s_01  s_02  s_03  s_04  s_05  c_01  c_02  c_03  a_01  a_02  a_03  ei_01  ei_02  ei_03  ei_04  ei_05  ei_06  ei_07  si_01  si_02  si_03  si_04  si_05  si_06  si_07  si_08  si_09  si_10  si_11  si_12  si_13  si_14  ci_01  ci_02  ci_03  ci_04  ci_05  ci_06  ci_07  ai_01  ai_02  ai_03  ai_04
0  Afghanistan  AFG  2000 4.18 0.25 69.83 61.97 73.22 75.05 64.24 75.39 68.60 59.19 46.47 59.34 89.75 73.72 66.77 87.95 71.97  56.79   0.01   0.01  44.91   0.01   2.49   0.01  69.30  81.03  76.79  39.50  70.18  67.71  77.73  87.45  38.52  93.27  89.84  64.46  53.77  59.83  42.57  82.71  89.19  90.31  69.52  73.69  78.22  64.41  69.22  79.67  97.10
1  Afghanistan  AFG  2001 4.19 0.25 70.28 59.53 74.13 78.67 65.75

In [26]:
# --- TABLE 2: Latest-Year Snapshot ---

latest_year = wri_clean['year'].max()
wri_snapshot = wri_clean[wri_clean['year'] == latest_year].copy()

for score in COMPOSITE_SCORES:
    wri_snapshot[f'{score}_rank'] = wri_snapshot[score].rank(ascending=False).astype(int)

print(f"\n{'=' * 70}")
print(f"TABLE 2: LATEST-YEAR SNAPSHOT ({latest_year})")
print(f"{'=' * 70}")
print(f"Shape: {wri_snapshot.shape}")
print(f"Countries: {len(wri_snapshot)}")
print(f"\nTop 5 by overall WRI score:")
top5_cols = ['country', 'iso3', 'w', 'w_rank', 'e', 'v', 's', 'c', 'a']
print(wri_snapshot.nlargest(5, 'w')[top5_cols].to_string(index=False))


TABLE 2: LATEST-YEAR SNAPSHOT (2025)
Shape: (193, 254)
Countries: 193

Top 5 by overall WRI score:
    country iso3     w  w_rank     e     v     s     c     a
Philippines  PHL 46.56       1 39.99 54.20 50.10 58.54 54.30
      India  IND 40.73       2 35.99 46.10 34.56 54.08 52.43
  Indonesia  IDN 39.80       3 39.89 39.71 24.80 51.27 49.24
   Colombia  COL 39.26       4 31.54 48.86 47.85 50.87 47.91
     Mexico  MEX 38.96       5 50.08 30.31 44.39 12.53 50.07


In [27]:
# --- TABLE 3: Country-level summary statistics ---

wri_decomp = wri_clean.groupby(['country', 'iso3']).agg(
    # Long-run average risk level per dimension
    w_mean=('w', 'mean'),
    e_mean=('e', 'mean'),
    v_mean=('v', 'mean'),
    s_mean=('s', 'mean'),
    c_mean=('c', 'mean'),
    a_mean=('a', 'mean'),
    
    # Historical volatility — how stable or erratic each country's risk profile has been
    w_std=('w', 'std'),
    e_std=('e', 'std'),
    v_std=('v', 'std'),
    
    # Most recent observation for current-state context
    w_latest=('w', 'last'),
    e_latest=('e', 'last'),
    v_latest=('v', 'last'),
    
    # Time series metadata
    year_count=('year', 'count'),
    first_year=('year', 'min'),
    last_year=('year', 'max')
).round(3)

wri_decomp['ev_ratio'] = (wri_decomp['e_mean'] / wri_decomp['v_mean'].replace(0, np.nan)).round(3)

print(f"\n{'=' * 70}")
print(f"TABLE 3: RISK DECOMPOSITION SUMMARY (per country)")
print(f"{'=' * 70}")
print(f"Shape: {wri_decomp.shape}")

print(f"\nHighest Exposure-to-Vulnerability ratio (resilient despite exposure):")
top_ev = wri_decomp.nlargest(5, 'ev_ratio')[['w_mean', 'e_mean', 'v_mean', 'ev_ratio']]
print(top_ev.to_string())

print(f"\nLowest Exposure-to-Vulnerability ratio (fragile despite low exposure):")
bot_ev = wri_decomp.nsmallest(5, 'ev_ratio')[['w_mean', 'e_mean', 'v_mean', 'ev_ratio']]
print(bot_ev.to_string())


TABLE 3: RISK DECOMPOSITION SUMMARY (per country)
Shape: (193, 16)

Highest Exposure-to-Vulnerability ratio (resilient despite exposure):
                               w_mean  e_mean  v_mean  ev_ratio
country                  iso3                                  
Japan                    JPN    22.12   43.85   11.29      3.88
China                    CHN    34.43   64.42   18.75      3.44
Australia                AUS    19.80   31.59   12.49      2.53
United States of America USA    25.37   39.77   16.34      2.44
Mexico                   MEX    36.41   50.03   26.69      1.87

Lowest Exposure-to-Vulnerability ratio (fragile despite low exposure):
                               w_mean  e_mean  v_mean  ev_ratio
country                  iso3                                  
Mali                     MLI     2.08    0.08   54.37      0.00
Niger                    NER     1.99    0.07   57.02      0.00
Sao Tome and Principe    STP     0.52    0.02   13.80      0.00
Burkina Faso         

In [28]:
# --- Export all three tables ---
OUTPUT_PATH = "./"

wri_core_panel.to_csv(f"{OUTPUT_PATH}wri_core_panel.csv", index=False)
wri_snapshot.to_csv(f"{OUTPUT_PATH}wri_snapshot_{latest_year}.csv", index=False)
wri_decomp.to_csv(f"{OUTPUT_PATH}wri_decomposition_summary.csv")

# Also save the column category definitions as a reference file
col_defs = pd.DataFrame({
    'category': ['id'] * len(ID_COLS) + 
                ['composite'] * len(COMPOSITE_SCORES) +
                ['component'] * len(COMPONENT_COLS) +
                ['domain'] * len(DOMAIN_COLS) +
                ['norm_indicator'] * len(NORM_COLS) +
                ['base_indicator'] * len(BASE_COLS),
    'column': ID_COLS + COMPOSITE_SCORES + COMPONENT_COLS + 
              DOMAIN_COLS + NORM_COLS + BASE_COLS
})
col_defs.to_csv(f"{OUTPUT_PATH}column_definitions.csv", index=False)

# Save the full cleaned dataset too (for when we need Norm/Base detail)
wri_clean.to_csv(f"{OUTPUT_PATH}wri_full_clean.csv", index=False)

print(f"\n{'=' * 70}")
print("FILES EXPORTED")
print(f"{'=' * 70}")
print(f"1. wri_core_panel.csv              — {wri_core_panel.shape[0]:,} rows × {wri_core_panel.shape[1]} cols")
print(f"2. wri_snapshot_{latest_year}.csv       — {wri_snapshot.shape[0]:,} rows × {wri_snapshot.shape[1]} cols")
print(f"3. wri_decomposition_summary.csv   — {wri_decomp.shape[0]:,} rows × {wri_decomp.shape[1]} cols")
print(f"4. column_definitions.csv          — Reference: which columns belong to which tier")
print(f"5. wri_full_clean.csv              — Complete dataset with all 248 columns")
print(f"\nPhase 1 complete. These tables feed directly into Phase 2 (Feature Engineering).")


FILES EXPORTED
1. wri_core_panel.csv              — 5,018 rows × 52 cols
2. wri_snapshot_2025.csv       — 193 rows × 254 cols
3. wri_decomposition_summary.csv   — 193 rows × 16 cols
4. column_definitions.csv          — Reference: which columns belong to which tier
5. wri_full_clean.csv              — Complete dataset with all 248 columns

Phase 1 complete. These tables feed directly into Phase 2 (Feature Engineering).
